# RAG 평가 개요
- RAG 평가란 RAG 시스템이 주어진 입력에 대해 얼마나 효과적으로 관련 정보를 검색하고, 이를 기반으로 정확하고 유의미한 응답을 생성하는지를 측정하는 과정이다. 
- **평가 요소**
    - **검색 단계 평가**
        - 입력 질문에 대해 검색된 문서나 정보의 관련성과 정확성을 평가.
    - **생성 단계 평가**
        - 검색된 정보를 기반으로 생성된 응답의 품질, 정확성등을 평가.
- **평가 방법**
    - 온/오프라인 평가
        1. **오프라인 평가**
            - 미리 준비된 데이터셋을 활용하여 RAG 시스템의 성능을 측정한다.
        2. **온라인 평가**
            - 실제 사용자 트래픽과 피드백을 기반으로 시스템의 실시간 성능을 평가한다.
    - 정량적/정성적 평가
        1. 정량적 평가
            - 자동화된 지표를 사용하여 생성된 텍스트의 품질을 평가한다.
        2. 정성적 평가
            - 전문가나 일반 사용자가 직접 생성된 응답의 품질을 평가하여 주관적인 지표를 평가한다.

# [RAGAS](https://www.ragas.io/)
- RAGAS는 RAG 파이프라인을 **정량적 으로 평가하는** 오픈소스 프레임 워크이다. 
- RAGAS 문서: https://docs.ragas.io/en/stable/
## 설치
- `pip install ragas`

In [2]:
import ragas

ragas.__version__

'0.2.15'

## RAGAS 평가 지표 개요
![ragas_score](figures/ragas_score.png)
- **Generation**
    - llm 모델이 생성한 답변에 대한 평가 지표들.
    - **Faithfulness(신뢰성)**
        -  생성된 답변과 검색된 문서(context)간의 관련성을 평가하는 지표
        -  생성된 답변이 주어진 문맥(context)에 얼마나 충실한지를 평가하는 지표로 할루시네이션에 대한 평가로 볼 수있다.
    - **Answer relevancy(답변 적합성)**
        - 생성된 답변과 사용자의 질문간의 관련성을 평가하는 지표
        - 생성된 답변이 사용자의 질문과 얼마나 관련성이 있는지를 평가하는 지표.
- **Retrieval**
    -  질문에 대해 검색한 문서(context)들에 대한 평가
    -  **Context Precision(문맥 정밀도)**
        -  검색된 문서(context)들 중 질문과 관련 있는 것들이 **얼마나 상위 순위에 위치하는지** 평가하는 지표.
    -  **Context Recall(문맥 재현률)**
        -  검색된 문서(context)가 정답(ground-truth)의 정보를 얼마나 포함하고 있는지 평가하는 지표.
- 이러한 지표들은 RAG 파이프라인의 성능을 다각도로 평가하는 데 활용된다.
![RAGAS_score2](figures/RAGAS_score2.png)

## 주요 평가지표
### Generation 평가
- LLM이 생성한 답변에 대한 평가
  
#### Faithfulness (신뢰성)
- 생성된 답변이 얼마나 주어진 검색 문서들(context)를 잘 반영해서 생성되었는지 평가한다. 할루시네이션에 대한 평가라고 할 수 있다. 
- 점수범위: **0 ~ 1** (1에 가까울수록 좋음)
- 답변에 포함된 모든 주장이 context에서 얼마나 추출 가능한지를 확인한다.

##### 평가 방법
1. Answer에서 주장 구문(claim statement)들을 생성(추출)한다. (주장이란, 질문(user input)과 관련된 내용)
    - 예) 
        - **질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요? 
        - **LLM 답변**: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 3000만명이다.
2. 각 주장들을 context로 부터 추론 가능한지 판단한다. 이를 바탕으로 faithfulness 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다. .... 한국의 인구는 5000만명이고 서울에 1000만이 살고 있다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 추론가능한 주장.
            - 한국의 인구는 3000만명이다. -> context에서 추론 불가능한 주장.
3. **Faithfulness score** 를 계산한다. 총 주장 수 중에서 context로 부터 추론가능한 주장의 개수.    
    - 예)
        - Faithfulness Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 유추할 수있다.)
    - LLM 답변에서 주장을 추출 하는 것과 각 주장이 context에서 추론 가능한 지를 판단하는 것은 LLM 을 활용한다.
- 공식
    $$
    \text{Faithfulness Score}\;=\;\cfrac{\text{주어진\;context\;에서\;추론할\;수\;있는\;주장의\;개수}}{\text{총\;주장\;개수}}
    $$

#### Answer relevancy (답변 적합성)
- 생성된 답변이 질문(user input)에 얼마나 잘 부합하는 지를 평가한다.
- 점수 범위: -1~1 (1에 가까울수록 좋음)
- LLM이 생성한 답변을 기반으로 질문들을 생성한다. 이렇게 생성한 질문들과 실제 질문(user input)의 embedding vector 간의 **코사인 유사도**를 측정한다.

##### 평가방법
1. LLM이 생성한 답변을 기반으로 질문들을 생성한다.
    - 예) 
        - **LLM** 답변: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
2. 실제 질문과 생성한 질문간의 코사인 유사도를 측정한다. 그 평균이 최종 점수가 된다.
    - 예)
        - **실제 질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요?
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
- 공식
  $$
    \cfrac{1}{N} \sum_{i=1}^{N} \text{cosine\_similarity}(q_{\text{user}_{_i}}, q_{\text{generated}})
  $$

### Retrieval 평가
User input에 대해 Vector store에서 검색한 context에 대한 평가

#### Context Precision
- 사용자가 질문(query)에 대해 검색된 문서들이 답변 생성에 얼마나 유용한지를 평가한다.
- 검색된 K개의 문서(context)들 중 **질문과 관련 있는 문서들이 얼마나 상위 순위**에 있는 지로 평가.
- 점수 범위: 0~1 (1에 가까울수록 좋음)


##### 평가방법

- 공식
$$
 \text{Context\;Precision@K} = \frac{\sum_{k=1}^{K} \left( \text{Precision@k} \times v_k \right)}{\ 상위\;K개\;결과에서의\;관련\;항목\;수}
$$
$$
 \text{Precision@k} = \frac{\text{True\;positive@k}}{(\text{True\;positive@k} + \text{False\;positive@k})} \\
$$
- $\text{True Positive@k}$: 상위 k개의 문서 중 질문과 관련있는 문서의 개수
- $\text{False Positive@k}$: 상위 k개의 문서 중 질문과 관련없는 문서의 개수
- $\text{Precision@k}$: 상위 k개의 문서 중 질문과 관련된 문서들이 차지하는 비율
- K: 검색한 context(문서)의 개수(chuck 수)
- $v_k$: 질문과의 context간의 관련성 여부로 0 또는 1. (0: 관련 없음, 1: 관련 있음)

##### 예시
- 질문과 context 관련성 예
    - 질문: 한국의 수도는 어디이고 인구는 얼마나 되나요?
    - 높은 정밀도 context(관련성 높은 문서의 예)
        - 한국의 수도는 서울이고 인구는 5000명 입니다. 
        - 한국의 수도는 서울입니다.
        - 한국은 동아시아에 위치해 있는 국가로 수도는 서울입니다.
        - 한국의 인구는 5000만명 입니다.
    - 낮은 정밀도 context(관련성 낮은 문서의 예)
        - 한국은 동아시아에 위치한 국가입니다.
        - 한국의 K-pop은 전 세계적으로 유명합니다.
        - 비빔밥, 불고기는 한국의 대표적인 음식입니다.
    - **높은 정밀도의 context이 상위 순위에 위치했으면 높은 점수를 받는다.**
- 점수 계산의 예
  - 상위 5개의 검색 결과 중 1번째, 3번째, 4번째 문서가 관련이 있다고 가정
    ```bash
    Precision@1 = 1/1 = 1.0    # True positive@1/(True positive@1 + False positive@1).  1/1(1번 문서 계산 시에는 1개 문서만 있으므로 분모가 1이 된다.)
    Precision@2 = 1/2 = 0.5     
    Precision@3 = 2/3 ≈ 0.67    
    Precision@4 = 3/4 = 0.75
    Precision@5 = 3/5 = 0.6
    ```
- vk의 값
    - 1번째: $v_1 = 1$
    - 2번째: $v_2 = 0$
    - 3번째: $v_3 = 1$
    - 4번째: $v_4 = 1$
    - 5번째: $v_5 = 0$

- Context Precision@5
$$
\text{Context\;Precision@5} = \frac{(1.0 \times 1) + (0.5 \times 0) + (0.67 \times 1) + (0.75 \times 1) + (0.6 \times 0)}{3} = \frac{1.0 + 0 + 0.67 + 0.75 + 0}{3} ≈ 0.807
$$

#### Context Recall (문맥 재현률)
- 검색된 문서(context)가 얼마나 정답(ground-truth)의 정보를 포함있는 지 평가하는 지표
- 점수 범위: 0~1 (1에 가까울수록 좋음)
- **정답(ground truth)의 각 주장(claim)이 검색된 context와 얼마나 일치**하는지 계산함.

##### 평가방법
1. **정답**에서 주장 문장(claim statement)들을 생성(추출)한다.
    - 예) 
        - **정답**: 한국의 수도는 서울이고 인구수는 5000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 5000만명이다.
2. 각 주장 문장(claim statement)의 정보를 검색된 context들에서 찾을 수 있는지 판별한다. 이를 바탕으로 context recall 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 찾을 수 있다.
            - 한국의 인구는 5000만명이다. -> context에서 찾을 수 없다.
3. **Context Recall Score** 를 계산한다. 총 주장 수 중에서 context로 부터 찾을 수 있는 주장의 개수.
    - 예)
        - Context Recall Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 찾을 수 있다.)

- 공식
    $$
    \text{Context Recall Score}\;=\;\cfrac{\text{GT의\;주장\;중\;주어진\;context\;에서\;찾을\;수\;있는\;주장의\;개수}}{\text{GT의\;총\;주장\;개수}}
    $$ 

# RAGAS 평가 실습

In [ ]:
# %pip install ragas

In [ ]:
# ### Vector Store 연결 1

# from langchain_community.document_loaders import TextLoader
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_chroma import Chroma


# COLLECTION_NAME = "olympic_info"
# PERSIST_DIRECTORY = "vector_store/olympic_info"
# DOC_PATH = 'data/olympic.txt'

# loader = TextLoader(DOC_PATH, encoding='utf-8')
# splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
#     model_name="gpt-4o-mini", chunk_size=500, chunk_overlap=100
# )
# docs = loader.load_and_split(splitter)

# vector_store = Chroma.from_documents(
#     documents=docs,
#     embedding=embedding_model,
#     collection_name=COLLECTION_NAME,
#     persist_directory=PERSIST_DIRECTORY
# )

In [ ]:
# Vector Store 연결 2

# COLLECTION_NAME = "olympic_info"
# PERSIST_DIRECTORY = "vector_store/olympic_info"

# vector_store = Chroma(
#     embedding_function=embedding_model,
#     collection_name=COLLECTION_NAME,
#     persist_directory=PERSIST_DIRECTORY
# )

In [1]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI(model="gpt-4.1-mini")
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large"
)


In [4]:
## ChromaDB와 연결 하고 vector store 생성
from langchain_chroma import Chroma

COLLECTION_NAME = 'olympic_info'
PERSIST_DIRECTORY = 'vector_store/chroma/olympic_info'

vector_store = Chroma(
    embedding_function=embedding_model,
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIRECTORY
)
print(vector_store._collection.count())

0


In [6]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

loader = TextLoader("data/olympic.txt", encoding="utf-8")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

docs = loader.load_and_split(splitter)
# vector_store.add_documents(docs)

['c83db18f-9216-4612-9c1b-3baf17b85336',
 '035590ed-7ab2-48b8-a658-9279a9ffe5b6',
 '1f25ceb5-c5f0-43c4-bf2f-6330e8fe8cde',
 'a6a903d5-69f1-44e6-9de8-cfc3ecc511df',
 'd1824c93-1720-46af-90ec-05c9ca7bce9b',
 'b7d50b03-a653-40c1-b976-ef8fdd0b5275',
 'b03607e1-2afa-438f-b19a-0f8f559f2296',
 'c4bb2c00-2f0a-447c-b957-c0da25a6e23c',
 '9ff35cd3-267e-4758-be72-ce1350cfbf42',
 '3a6bf756-0bbf-45f3-8b86-3442c32f9fca',
 '3186f8a4-7253-42a7-87bb-6a2f4d3ef67e',
 'ef27eb1d-25f0-4f95-965d-3428f364e584',
 'b897e070-16e1-47f4-876d-7f2367d65b09',
 '7bfeb328-cb0e-40f8-af4e-8cb17d449433',
 'a3693ed1-bada-4c44-83bb-e985bc7a5c11',
 '128bbea1-a437-4dde-b517-c96cdff83782',
 '60bc631e-c6f6-4e9b-8efa-563eb79319b8',
 '6377d79d-848b-4458-b44a-8d3e39276a77',
 '1866888b-3f5d-4b66-a328-081d3b7d9578',
 '82317ad9-16df-40d7-986d-14a09d77e0c6',
 'ff8fcbbc-3215-4a31-999c-074023838915',
 '72dd0036-dcb7-4794-b25e-fd12490c2333',
 '2955d1b3-9134-4a25-8c9e-0f6f2164241c',
 'c9cde634-e21a-48de-b15d-1bf4a5ce0b2d',
 '9d3e990c-b460-

In [7]:
print(vector_store._collection.count())

61


## RAG Chain 구성
- Vector Store 연결
- Chain 응답 결과
    - 평가를 위해 **Vector Store에서 검색한 context**들과 **LLM 응답**이 출력되도록 chain을 구성한다.

In [8]:
from langchain.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

retriever = vector_store.as_retriever()

template = """# Instruction:
당신은 정확한 정보 제공을 우선시하는 인공지능 어시스턴트입니다.
주어진 Context에 포함된 정보만 사용해서 질문에 답변하세요.
Context에 질문에 대한 명확한 정보가 있는 경우 그 내용을 바탕으로 답변하세요.
Context에 질문에 대한 명확한 정보없을 경우 "정보가 부족해서 답을 알 수 없습니다." 라고 대답합니다.
절대 Context에 없는 내용을 추측하거나 일반 상식을 이용해 답을 만들어서 대답하지 않습니다.

# Context:
{context}

# 질문:
{query}
"""
prompt_template = PromptTemplate(template=template)

In [9]:
def format_docs(src_dict:dict)->str:
    """
    Vector_store에 조회한 문서들에서 내용(page_content)만 추출해서 str으로 합쳐서 반환.
    
    Args:
        src_dict(dict): {"context":list[Document],  "query":"사용자질문"}
    Returns:
        str: 각 문서의 내용을 "\n\n"으로 연결한 string
    """
    docs = src_dict["context"]
    return "\n\n".join(doc.page_content for doc in docs)

def format_docs_list(src_dict:dict)->list[str]:
    """
    Vector_store에 조회한 문서들에서 내용(page_content)만 추출해서 list로 묶어서 반환.
    
    Args:
        src_dict(dict): {"context":list[Document],  "query":"사용자질문"}
    Returns:
        list[str]: 각 문서의 내용을 담은 list
    """
    return [doc.page_content for doc in src_dict["context"]]

In [14]:
docs = retriever.invoke("올림픽 논란")
docs


[Document(id='c83db18f-9216-4612-9c1b-3baf17b85336', metadata={'source': 'data/olympic.txt'}, page_content='올림픽'),
 Document(id='128bbea1-a437-4dde-b517-c96cdff83782', metadata={'source': 'data/olympic.txt'}, page_content='하계올림픽'),
 Document(id='ff8fcbbc-3215-4a31-999c-074023838915', metadata={'source': 'data/olympic.txt'}, page_content='패럴림픽'),
 Document(id='b03607e1-2afa-438f-b19a-0f8f559f2296', metadata={'source': 'data/olympic.txt'}, page_content='고대올림픽')]

In [ ]:
format_docs({"context":docs})
format_docs_list({"context":docs})

In [15]:
# 평가를 위한 Rag Chain을 구성
## chain의 응답: LLM 응답, Retriever가 검색한 context들 
rag_chain = (
    RunnablePassthrough()  # dict | dict 를 LCEL로 연결하기 위해 하는일 없는 Runnable(RunnablePassthrough) 을 추가
    | {"context":retriever, "query":RunnablePassthrough()} # retrieve
    | {
        "source_context":format_docs_list, # list[Document] -> list[str]
        "llm_answer": {
                        "context":format_docs, # list[Document] -> str(문서\n\n문서\n\n...)
                        "query": lambda x : x["query"]    # query만 추출
                    } | prompt_template | model | StrOutputParser()
    } # 응답처리 - 입력: {"context":검색문서들, "query":"사용자질문"} 
      #          - 출력: {"source_context":검색한 문서들(list[str]), "llm_answer":LLM응답(str)}
      #             -> 응답처리 출력이 chain의 최종 출력
    
)

In [16]:
user_input = "국제 올림픽 위원회에 대해서 설명해주세요."
response = rag_chain.invoke(user_input)

In [17]:
print(type(response), response.keys())

<class 'dict'> dict_keys(['source_context', 'llm_answer'])


In [18]:
llm_answer = response['llm_answer']
print(llm_answer)

국제 올림픽 위원회(IOC)는 모든 올림픽 활동을 통솔하는 단체로서, 올림픽 개최 도시 선정, 계획 감독, 종목 변경, 스폰서 및 방송권 계약 체결 등의 권리가 있습니다. 올림픽 활동은 많은 국가, 국제 경기 연맹과 협회, 미디어 파트너와 협력하며, 선수, 직원, 심판 등 모든 관련자가 올림픽 헌장을 지키는 것을 포함합니다. IOC는 국제경기연맹(IF), 국가 올림픽 위원회(NOC), 올림픽 조직 위원회(OCOG)를 통해 올림픽을 구성하고 관리합니다. 또한 IOC 위원들은 때때로 여러 비판을 받기도 했으며, 대표적인 예로 위원장 에이버리 브런디지와 후안 안토니오 사마란치가 있습니다.


In [19]:
context = response["source_context"]
context

['국제 올림픽 위원회\n올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기 • 선수, 직원, 심판, 모든 사람과 기관이 올림픽 헌장을 지키는 것을 말한다. 국제올림픽위원회(IOC)는 모든 올림픽 활동을 통솔하는 단체로서, 올림픽 개최 도시 선정, 계획 감독, 종목 변경, 스폰서 및 방송권 계약 체결 등의 권리가 있다. 올림픽 활동은 크게 세 가지로 구성된다.\n- 국제경기연맹(IF)은 국제적인 규모의 경기를 관리, 감독하는 기구이다. 예를 들어서 국제 축구 연맹(FIFA)는 축구를 주관하며, 국제 배구 연맹(FIVB)은 배구를 주관하는 기구이다. 올림픽에는 현재 35개의 국제경기연맹이 있고 각 종목을 대표한다. (이 중에는 올림픽 종목은 아니지만 IOC의 승인을 받은 연맹도 있다.)',
 '- 국가 올림픽 위원회(NOC)는 각국의 올림픽 활동을 감독하는 기구이다. 예를 들어서 대한 올림픽 위원회(KOC)는 대한민국의 국가 올림픽 위원회이다. 현재 IOC에 소속된 국가 올림픽 위원회는 205개이다.\n- 올림픽 조직 위원회(OCOG)는 임시적인 조직으로 올림픽의 총체적인 것(개막식, 페막식 등)을 책임지기 위해 구성된 조직이다. 올림픽 조직 위원회는 올림픽이 끝나면 해산되며 최종보고서를 IOC에 제출한다.\n올림픽의 공식언어는 프랑스어와 영어와 개최국의 공용어이다. 모든 선언(예를 들어서 개막식 때 각국 소개를 할 때)들은 세 언어가 모두 나오거나 영어나 프랑스어 중에서 한 언어로만 말하기도 한다. 개최국의 공용어가 영어나 프랑스어가 아닐 때는 당연히 그 나라의 공용어도 함께 나온다.',
 '올림픽은 국제경기연맹(IF), 국가 올림픽 위원회(NOC), 각 올림픽의 위원회(예-벤쿠버동계올림픽조직위원회)로 구성된다. 의사 결정 기구인 IOC는 올림픽 개최 도시를 선정하며, 각 올림픽 대회마다 열리는 올림픽 종목도 IOC에서 결정한다. 올림픽 경기 개최 도시는 경기 축하 의식이 올림픽 헌장에 부합하도록 조직하고 기금을 마련해야 한다. 

## 평가

In [20]:
# 평가데이터셋: user_input, response(정답)
user_input = "국제 올림픽 위원회에 대해 설명해주세요."
reference = "국제올림픽위원회(IOC)는 올림픽 활동을 통솔하는 기구로, 올림픽 개최 도시 선정, 계획 감독, 종목 변경, 스폰서 및 방송권 계약 체결 등의 역할을 수행한다."

resp = rag_chain.invoke(user_input)
retrieved_context = resp['source_context']  # RAG에서 검색된 문서
response = resp['llm_answer']  # RAG의 답변

In [21]:
from ragas import SingleTurnSample, EvaluationDataset

# RAGAS의 평가 데이터셋의 구성
# - user_input: 사용자 입력(질문)
# - retrieved_context: RAG에서 검색한 context(문서)들
# - response: llm의 출력(응답)
# - reference: 정답.

eval_sample1 = SingleTurnSample(  # 1개의 평가 데이터을 생성할 때 사용.
    user_input=user_input,
    retrieved_contexts=retrieved_context,
    response=response,
    reference=reference
)
# 평가용 Dataset을 생성
eval_dataset = EvaluationDataset(samples=[eval_sample1])

In [22]:
eval_dataset

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response', 'reference'], len=1)

In [23]:
# eval_dataset을 DataFrame으로 변환
eval_dataset.to_pandas()

,user_input,retrieved_contexts,response,reference
0,국제 올림픽 위원회에 대해 설명해주세요.,"[국제 올림픽 위원회\n올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 •...","국제 올림픽 위원회(IOC)는 모든 올림픽 활동을 통솔하는 단체로서, 올림픽 개최 ...","국제올림픽위원회(IOC)는 올림픽 활동을 통솔하는 기구로, 올림픽 개최 도시 선정,..."


In [ ]:
##########################
# 평가
from ragas.metrics import (
    LLMContextRecall, # Context recall을 계산하는 **평가함수**
    LLMContextPrecisionWithReference, # Context Precision
    Faithfulness,     # Faithfulness 
    AnswerRelevancy,  # AnswerRelevancy
)
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
# Langchain으로 생성된 LLM 모델, 임베딩모델 객체를 RAGAS에서 사용할 있도록 wrapping하는 클래스.
# 모든 평가 지표들이 LLM 모델을 사용한다. 

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv
load_dotenv()

In [26]:
# 평가 함수(객체)에 넣어줄 LLM, Embedding model 생성

llm_model = ChatOpenAI(model = "gpt-4.1-mini")
e_model = OpenAIEmbeddings(model = "text-embedding-3-large")

eval_model = LangchainLLMWrapper(llm_model)
eval_embedding_model = LangchainEmbeddingsWrapper(e_model)

## 평가 객체를 생성 -> list로 묶어줌
metrics = [
    LLMContextRecall(llm=eval_model),
    LLMContextPrecisionWithReference(llm=eval_model),
    Faithfulness(llm=eval_model),
    AnswerRelevancy(llm=eval_model)
]

# 평가지표 객체를 이용해서 평가
eval_result = evaluate(dataset=eval_dataset, metrics=metrics)

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

In [30]:
# context Recall : 검색 context와 정답간의 연관성
# context Precison : 검색 context와 사용자 질문간의 연관성
# Faithfulness : LLM 답변과 검색 context간의 연관성
# Answer relevancy : LLM 답변과 사용자 질문간의 연관성

print(eval_result)

{'context_recall': 1.0000, 'llm_context_precision_with_reference': 0.8333, 'faithfulness': 0.8235, 'answer_relevancy': 0.8300}


In [31]:
eval_result.to_pandas()

,user_input,retrieved_contexts,response,reference,context_recall,llm_context_precision_with_reference,faithfulness,answer_relevancy
0,국제 올림픽 위원회에 대해 설명해주세요.,"[국제 올림픽 위원회\n올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 •...","국제 올림픽 위원회(IOC)는 모든 올림픽 활동을 통솔하는 단체로서, 올림픽 개최 ...","국제올림픽위원회(IOC)는 올림픽 활동을 통솔하는 기구로, 올림픽 개최 도시 선정,...",1.0,0.833333,0.823529,0.82999


# 평가 데이터 셋 만들기

1. 문서를 Loading하고 split하여 Context 들을 만든다.
2. Context들 중 평가에 사용할 것을 Random하게 선택한다.
3. 선택된 context를 기반으로 LLM 모델을 이용해서 질문과 답변을 생성한다. 
4. 생성된 질문과 답변을 검토하여 품질을 높인다. 
   - 질문과 답변 자체가 맞는지 검사한다.
   - 나올만한 질문인 지 확인한다.

- **생성된 질문-답변의 질이 낮으면 좋은 평가를 할 수 없다.**
    - 질문-답변 생성시 사용하는 LLM 모델은 성능이 좋은 것을 사용해야 한다. 
    - 생성된 질문-답변을 사람이 검토해서 품질을 더 높여야 한다.

## 데이터셋에 포함될 내용
- Dataset을 생성할 때는 다음 항목이 들어가야 한다.
    -  **user_input**: 질문. `string`
    -  **reference**: 정답. `string`
    -  qa_context : context. `str` - QA를 만들 때 어떤 context를 사용했는지 값. 생성된 데이터셋을 확인하기 위한 것으로 **평가시에는 사용하지 않는다.**
-  평가할 때 추가할 것
    - **response**: RAG 시스템에서 질문에 대해 LLM 모델이 생성한 답변. `string`
    - **retrieved_contexts**: RAG 시스템에서 질문에 대해 Vector DB에서 검색한 context들. `list(string)`

In [32]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser,StrOutputParser

from langchain import hub
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.documents import Document

from ragas import EvaluationDataset, RunConfig, evaluate
from ragas.metrics import (
    LLMContextRecall, Faithfulness, LLMContextPrecisionWithReference, AnswerRelevancy
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from pydantic import BaseModel, Field

import random
import pandas as pd

from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# Context를 바탕으로 질문과 답변ㅇ르 생성
# 생성된 질문을 rag_chain에 넣어서 context와 llm 응답을 생성.

In [33]:
# context 생성
DOC_PATH = "data/olympic.txt"

loader = TextLoader(DOC_PATH, encoding = "utf-8")
# Vector DB에 저장한 context와 동일하게 chunking(split) 한다.
splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 50)

docs = loader.load_and_split(splitter)
len(docs)

61

In [37]:
# docs 중에서 평가 data를 생성할 때 문서 K개를 추출
#### 실제 평가 dataset을 생성할 대는 모든 문서를 다 사용.
total_sample = 5	# 5개 context만 사용.

index_list = list(range(len(docs)))
random.shuffle(index_list)

eval_context = []	# 추출한 context들을 담을 list
while len(eval_context) < total_sample:
    index = index_list.pop()
    context = docs[index].page_content
    if len(context) > 200:	# 200글자 이상인 것만 사용.
        eval_context.append(context)

In [38]:
eval_context

['20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예를 들어 1904년 하계 올림픽 마라톤에서 우승한 미국 선수 토머스 J. 힉스는 코치에게서 스트리크닌과 브랜디를 받았다. 올림픽에서 약물을 과다 복용으로 사망한 사례도 한 번 있었다. 1960년 로마 대회 때 사이클 개인도로 경기 중에 덴마크 선수인 크누드 에네마르크 옌센이 자전거에서 떨어져서 사망했다. 검시관들의 조사에 의하면 그의 죽음의 원인은 암페타민 과다 복용이라고 했다. 이에 1960년대 중반부터 각 경기 연맹은 약물 복용을 금지하기 시작했으며 1967년에는 IOC도 약물 복용 금지에 동참했다.',
 '또한 20세기에 올림픽 운동이 발전함에 따라, IOC는 변화하는 세계의 사회 환경에 적응해야 했다. 이러한 변화의 예로는 얼음과 눈을 이용한 경기 종목을 다루는 동계 올림픽, 장애인이 참여하는 패럴림픽, 스페셜 올림픽, 데플림픽, 10대 선수들이 참여하는 유스 올림픽 등을 들 수 있다. 그 뿐만 아니라 IOC는 20세기의 변화하는 경제, 정치, 기술 환경에도 적응해야 했다. 그리하여 올림픽은 피에르 드 쿠베르탱이 기대했던 순수한 아마추어 정신에서 벗어나서, 프로 선수도 참가할 수 있게 되었다. 올림픽은 점차 대중 매체의 중요성이 커짐에 따라 올림픽의 상업화와 기업 후원을 놓고도 논란이 생겨났다. 또한 올림픽을 치르며 발생한 보이콧, 도핑, 심판 매수, 테러와 같은 수많은 일들은 올림픽이 더욱 굳건히 성장할 수 있는 원동력이 되었다.',
 '2002년에 열린 제114차 IOC 총회에서는 하계 올림픽 종목은 최대 28부문 301개 경기에 10,500명이 참가하는 것으로 제한하기로 결정했다.그 후 3년 뒤인 제117차 IOC 총회에서는 정식종목이었던 야구와 소프트볼을 정식 종목에서 제외시킨다. 이 결과에 대한 이견이 없었으므로 2012년 올림픽 때는 26개부문에서 경기가 열린다. 2016년과 2020년 올림픽 때는 럭비와 골프가 추가되어 다시 28개부문에서 경기가 열린다.\n

In [48]:
# Context -> 질문과 답변을 생성하는 Chain
# JsonOutputParser를 사용
## 스키마 생성.
class EvalSchema(BaseModel):
    user_input: str = Field(..., description="사용자 질문")	# 스키마 변수에 대한 설정 `...`: 필수
    reference : str = Field(..., description="user_input(사용자 질문)에 대한 정답")
    qa_context : str = Field(..., description="질문, 답변 쌍을 만들 때 참조한 context. 입력된 context를 수정하지 않고 그대로 넣는다.")
    
parser = JsonOutputParser(pydantic_object=EvalSchema)

template = """# Instruction:
당신은 RAG 평가를 위해 질문과 정답 쌍을 생성하는 인공지능 비서입니다.
다음 [Context] 에 문서가 주어지면 해당 문서를 기반으로 {num_questions}개 질문-정답 쌍을 생성하세요. 

질문과 정답을 생성한 후 Output Indicator의 format으로 출력합니다.
질문은 반드시 Context 문서에 있는 정보를 바탕으로 생성해야 합니다. Context에 없는 내용을 가지고 질문-정답을 절대 만들면 안됩니다.
올림픽에 관심있는 일반 사용자 관점의 자연스러운 질문을 작성합니다.
질문은 간결하게 작성합니다.
하나의 질문에는 한 가지씩만 내용만 작성합니다.
질문을 만들 때 "제공된 문맥에서", "문서에 설명된 대로", "주어진 문서에 따라" 또는 이와 유사한 말을 하지 마세요.
정답은 반드시 Context에 있는 정보를 바탕으로 작성합니다. 없는 내용을 추가하지 않습니다.
질문과 정답을 만들고 그 내용이 Context에 있는 항목인지 다시 한번 확인합니다.
생성된 질문-답변 쌍은 반드시 dictionary 형태로 정의하고 list로 묶어서 반환해야 합니다.
질문-답변 쌍은 반드시 {num_questions}개를 만들어야 합니다.

# Context:
{context}

# Output Indicator:
{format_instructions}
"""

prompt_template = PromptTemplate(
    template=template,
    partial_variables={"format_instructions" : parser.get_format_instructions()}
)
eval_model = ChatOpenAI(model = "gpt-4.1")	# 좋은 모델 써야함
eval_dataset_chain = prompt_template | eval_model | parser

In [50]:
eval_dataset_chain.invoke({"context" : eval_context[0], "num_questions" : 3})	# 내용이 맞는지 꼭 체크해봐야 함.

[{'user_input': '올림픽에서 약물 과다 복용으로 사망한 사례가 있었나요?',
  'reference': '1960년 로마 올림픽에서 덴마크 사이클 선수 크누드 에네마르크 옌센이 암페타민 과다 복용으로 사망한 사례가 있습니다.',
  'qa_context': '20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예를 들어 1904년 하계 올림픽 마라톤에서 우승한 미국 선수 토머스 J. 힉스는 코치에게서 스트리크닌과 브랜디를 받았다. 올림픽에서 약물을 과다 복용으로 사망한 사례도 한 번 있었다. 1960년 로마 대회 때 사이클 개인도로 경기 중에 덴마크 선수인 크누드 에네마르크 옌센이 자전거에서 떨어져서 사망했다. 검시관들의 조사에 의하면 그의 죽음의 원인은 암페타민 과다 복용이라고 했다. 이에 1960년대 중반부터 각 경기 연맹은 약물 복용을 금지하기 시작했으며 1967년에는 IOC도 약물 복용 금지에 동참했다.'},
 {'user_input': '토머스 J. 힉스가 1904년 올림픽 마라톤에서 복용한 약물은 무엇인가요?',
  'reference': '토머스 J. 힉스는 스트리크닌과 브랜디를 복용했습니다.',
  'qa_context': '20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예를 들어 1904년 하계 올림픽 마라톤에서 우승한 미국 선수 토머스 J. 힉스는 코치에게서 스트리크닌과 브랜디를 받았다. 올림픽에서 약물을 과다 복용으로 사망한 사례도 한 번 있었다. 1960년 로마 대회 때 사이클 개인도로 경기 중에 덴마크 선수인 크누드 에네마르크 옌센이 자전거에서 떨어져서 사망했다. 검시관들의 조사에 의하면 그의 죽음의 원인은 암페타민 과다 복용이라고 했다. 이에 1960년대 중반부터 각 경기 연맹은 약물 복용을 금지하기 시작했으며 1967년에는 IOC도 약물 복용 금지에 동참했다.'},
 {'user_input': 'IOC가 약물 복용 금지 방침을 도입한 해는 언제인가요?',
  'r

In [55]:
# context당 5개씩 질문을 생성
eval_dataset_list = []
num_questions = 5
for context in eval_context:
    eval_data = eval_dataset_chain.invoke({"context" : context, "num_questions" : num_questions})
    eval_dataset_list.extend(eval_data)

In [56]:
len(eval_dataset_list[0])

3

In [57]:
##### 생성된 질문 쌍을 반드시 검토해야함.#####
eval_dataset_list[3]

{'user_input': '운동선수들의 약물 복용이 금지되기 시작한 시기는 언제인가요?',
 'reference': '1960년대 중반부터 각 경기 연맹은 약물 복용을 금지하기 시작했습니다.',
 'qa_context': '20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예를 들어 1904년 하계 올림픽 마라톤에서 우승한 미국 선수 토머스 J. 힉스는 코치에게서 스트리크닌과 브랜디를 받았다. 올림픽에서 약물을 과다 복용으로 사망한 사례도 한 번 있었다. 1960년 로마 대회 때 사이클 개인도로 경기 중에 덴마크 선수인 크누드 에네마르크 옌센이 자전거에서 떨어져서 사망했다. 검시관들의 조사에 의하면 그의 죽음의 원인은 암페타민 과다 복용이라고 했다. 이에 1960년대 중반부터 각 경기 연맹은 약물 복용을 금지하기 시작했으며 1967년에는 IOC도 약물 복용 금지에 동참했다.'}

In [59]:
# DataFrame으로 생성
import pandas as pd
eval_df = pd.DataFrame(eval_dataset_list)
eval_df.head()

,user_input,reference,qa_context
0,1904년 하계 올림픽 마라톤에서 우승한 선수는 누구인가요?,미국 선수 토머스 J. 힉스가 우승했습니다.,"20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예를..."
1,1960년 로마 올림픽에서 사이클 경기 중 사망한 선수는 누구인가요?,덴마크 선수 크누드 에네마르크 옌센이 사망했습니다.,"20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예를..."
2,크누드 에네마르크 옌센의 사망 원인은 무엇인가요?,암페타민 과다 복용이 사망 원인입니다.,"20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예를..."
3,운동선수들의 약물 복용이 금지되기 시작한 시기는 언제인가요?,1960년대 중반부터 각 경기 연맹은 약물 복용을 금지하기 시작했습니다.,"20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예를..."
4,IOC가 약물 복용 금지에 동참한 연도는 언제인가요?,1967년에 IOC도 약물 복용 금지에 동참했습니다.,"20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예를..."


In [60]:
eval_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_input  25 non-null     object
 1   reference   25 non-null     object
 2   qa_context  25 non-null     object
dtypes: object(3)
memory usage: 732.0+ bytes


In [61]:
#### 평가할 RAG chain(rag_chain)을 이용해서 context 문서들과 llm 응답을 받아서
## 		평가 dataset을 완성.
# user_input -> (rag_chain) -> {llm_answer, source_context}
context_list = []	# source_context들을 저장할 list
response_list = []	# llm_answer들을 저장할 list

for user_input in eval_df["user_input"]:
    resp = rag_chain.invoke(user_input)
    context_list.append(resp["source_context"])
    response_list.append(resp["llm_answer"])


In [62]:
len(context_list), len(response_list)

(25, 25)

In [64]:
eval_df.loc[0, "user_input"]

'1904년 하계 올림픽 마라톤에서 우승한 선수는 누구인가요?'

In [65]:
response_list[0]

'1904년 하계 올림픽 마라톤에서 우승한 선수는 미국 선수 토머스 J. 힉스입니다.'

In [69]:
context_list[0]

['20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예를 들어 1904년 하계 올림픽 마라톤에서 우승한 미국 선수 토머스 J. 힉스는 코치에게서 스트리크닌과 브랜디를 받았다. 올림픽에서 약물을 과다 복용으로 사망한 사례도 한 번 있었다. 1960년 로마 대회 때 사이클 개인도로 경기 중에 덴마크 선수인 크누드 에네마르크 옌센이 자전거에서 떨어져서 사망했다. 검시관들의 조사에 의하면 그의 죽음의 원인은 암페타민 과다 복용이라고 했다. 이에 1960년대 중반부터 각 경기 연맹은 약물 복용을 금지하기 시작했으며 1967년에는 IOC도 약물 복용 금지에 동참했다.',
 '올림픽에서 약물 복용 양성 반응이 나와서 메달을 박탈당한 첫 번째 사례로는 1968년 하계 올림픽의 근대 5종 경기에 출전해 동메달을 딴 한스 군나르 리렌바르가 있다. 그는 경기 후 도핑검사 결과 알코올을 복용한 것으로 확인되어 메달을 박탈당했다. 도핑 양성 반응으로 메달을 박탈당한 것으로 가장 유명한 사람은 1988년 하계 올림픽 육상 100m 경기에서 금메달을 땄으나 도핑 검사 결과 스타노졸롤을 복용한 것으로 확인돼 금메달을 박탈당한 캐나다 선수인 벤 존슨이 있다. 이에 따라 금메달은 2위를 했던 칼 루이스가 대신 받았다.',
 '고대 올림피아 경기가 처음 열린 시점은 보통 기원전 776년으로 인정되고 있는데, 이 연대는 그리스 올림피아에서 발견된 비문에 근거를 둔 것이다. 이 비문의 내용은 달리기 경주 승자 목록이며 기원전 776년부터 4년 이후 올림피아 경기 마다의 기록이 남겨져 있다. 고대 올림픽의 종목으로는 육상, 5종 경기(원반던지기, 창던지기, 달리기, 레슬링, 멀리뛰기), 복싱, 레슬링, 승마 경기가 있었다. 전설에 따르면 엘리스의 코로이보스가 최초로 올림피아 경기에서 우승한 사람이라고 한다.',
 "근대올림픽\n고대 올림피아 경기를 제대로 구현한 최초의 시도는 혁명 시대의 프랑스에서 1796년부터 1798년까지 3년동안 실시했던 프랑스 국내 올림픽인 '

In [70]:
eval_df["response"] = response_list
eval_df["retrieved_contexts"] = context_list

In [71]:
eval_df.head()

,user_input,reference,qa_context,response,retrieved_contexts
0,1904년 하계 올림픽 마라톤에서 우승한 선수는 누구인가요?,미국 선수 토머스 J. 힉스가 우승했습니다.,"20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예를...",1904년 하계 올림픽 마라톤에서 우승한 선수는 미국 선수 토머스 J. 힉스입니다.,"[20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예..."
1,1960년 로마 올림픽에서 사이클 경기 중 사망한 선수는 누구인가요?,덴마크 선수 크누드 에네마르크 옌센이 사망했습니다.,"20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예를...",1960년 로마 올림픽에서 사이클 경기 중 사망한 선수는 덴마크 선수인 크누드 에네...,"[20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예..."
2,크누드 에네마르크 옌센의 사망 원인은 무엇인가요?,암페타민 과다 복용이 사망 원인입니다.,"20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예를...",크누드 에네마르크 옌센의 사망 원인은 암페타민 과다 복용입니다.,"[20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예..."
3,운동선수들의 약물 복용이 금지되기 시작한 시기는 언제인가요?,1960년대 중반부터 각 경기 연맹은 약물 복용을 금지하기 시작했습니다.,"20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예를...",운동선수들의 약물 복용이 금지되기 시작한 시기는 1960년대 중반부터입니다. Con...,"[20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예..."
4,IOC가 약물 복용 금지에 동참한 연도는 언제인가요?,1967년에 IOC도 약물 복용 금지에 동참했습니다.,"20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예를...",IOC가 약물 복용 금지에 동참한 연도는 1967년입니다.,"[20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예..."


In [ ]:
##### RAGAS의 EvaluationDataset 생성
# feature로 "user_input", "response", "retrieved_contexts", "reference"를 가진 DataFrame으로 부터
# EvaluationDataset 생성.
evaluation_dataset = EvaluationDataset.from_pandas(eval_df)
evaluation_dataset

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response', 'reference'], len=25)

In [ ]:
# 평가
eval_llm = LangchainLLMWrapper(ChatOpenAI(name="gpt-4.1-mini"))
eval_embedding = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))
metrics = [
    LLMContextPrecisionWithReference(llm=eval_llm),
    LLMContextRecall(llm=eval_llm),
    Faithfulness(llm=eval_llm),
    AnswerRelevancy(llm=eval_llm, embeddings=eval_embedding)
]

eval_result = evaluate(dataset=evaluation_dataset, metrics=metrics)

Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

In [86]:
eval_result

{'llm_context_precision_with_reference': 0.9622, 'context_recall': 0.9200, 'faithfulness': 0.7459, 'answer_relevancy': 0.5655}

In [87]:
eval_result.to_pandas()

,user_input,retrieved_contexts,response,reference,llm_context_precision_with_reference,context_recall,faithfulness,answer_relevancy
0,1904년 하계 올림픽 마라톤에서 우승한 선수는 누구인가요?,"[20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예...",1904년 하계 올림픽 마라톤에서 우승한 선수는 미국 선수 토머스 J. 힉스입니다.,미국 선수 토머스 J. 힉스가 우승했습니다.,0.833333,1.0,1.000000,0.724583
1,1960년 로마 올림픽에서 사이클 경기 중 사망한 선수는 누구인가요?,"[20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예...",1960년 로마 올림픽에서 사이클 경기 중 사망한 선수는 덴마크 선수인 크누드 에네...,덴마크 선수 크누드 에네마르크 옌센이 사망했습니다.,1.000000,1.0,1.000000,0.844608
2,크누드 에네마르크 옌센의 사망 원인은 무엇인가요?,"[20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예...",크누드 에네마르크 옌센의 사망 원인은 암페타민 과다 복용입니다.,암페타민 과다 복용이 사망 원인입니다.,0.833333,1.0,1.000000,0.545097
3,운동선수들의 약물 복용이 금지되기 시작한 시기는 언제인가요?,"[20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예...",운동선수들의 약물 복용이 금지되기 시작한 시기는 1960년대 중반부터입니다. Con...,1960년대 중반부터 각 경기 연맹은 약물 복용을 금지하기 시작했습니다.,0.833333,1.0,1.000000,0.727639
4,IOC가 약물 복용 금지에 동참한 연도는 언제인가요?,"[20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예...",IOC가 약물 복용 금지에 동참한 연도는 1967년입니다.,1967년에 IOC도 약물 복용 금지에 동참했습니다.,0.833333,1.0,1.000000,0.693414
5,동계 올림픽은 어떻게 시작되었나요?,[동계올림픽\n동계 올림픽은 눈과 얼음을 이용하는 스포츠들을 모아 이루어졌으며 하계...,"동계 올림픽은 눈과 얼음을 이용하는 스포츠로 구성되었으며, 기존 하계 올림픽에서 실...",얼음과 눈을 이용한 경기 종목을 다루는 동계 올림픽이 20세기에 올림픽 운동의 발전...,1.000000,0.0,1.000000,0.547218
6,장애인이 참가할 수 있는 올림픽은 무엇인가요?,[패럴림픽(Paralympic)은 신체·감각 장애가 있는운동 선수가 참가하는 국제 ...,장애인이 참가할 수 있는 올림픽은 패럴림픽입니다.,"장애인이 참여하는 올림픽에는 패럴림픽, 스페셜 올림픽, 데플림픽이 있습니다.",1.000000,1.0,1.000000,0.738205
7,유스 올림픽에는 어떤 사람들이 참가하나요?,[올림픽은 거의 모든 국가가 참여할 정도로 규모가 커졌다. 하계 올림픽은 33개의 ...,정보가 부족해서 답을 알 수 없습니다.,유스 올림픽에는 10대 선수들이 참가합니다.,1.000000,0.0,0.000000,0.000000
8,올림픽에 프로 선수들도 참가할 수 있게 된 이유는 무엇인가요?,[현대 올림픽에서는 프로 선수의 참가 불허가 많은 분쟁을 가져왔다. 1912년 하계...,올림픽에 프로 선수들도 참가할 수 있게 된 이유는 20세기에 올림픽 운동이 발전함에...,20세기의 변화하는 환경에 적응하면서 올림픽은 아마추어 정신에서 벗어나 프로 선수도...,1.000000,1.0,1.000000,0.625986
9,올림픽에서 논란이 되었던 사건들은 무엇이 있나요?,[올림픽에서 약물 복용 양성 반응이 나와서 메달을 박탈당한 첫 번째 사례로는 196...,올림픽에서 논란이 되었던 사건들은 다음과 같습니다.\n\n1. 도핑 관련 사건 \...,"올림픽에서는 보이콧, 도핑, 심판 매수, 테러와 같은 사건들이 논란이 되었습니다.",1.000000,1.0,0.833333,0.746748


## 생성된 평가 dataset을 Huggingface에 save/load

In [89]:
from huggingface_hub import login
import os

HF_KEY = os.getenv("HUGGINGFACE_API_KEY")
login(HF_KEY)

In [91]:
# %pip install datasets
from datasets import Dataset

eval_dataset = Dataset.from_pandas(eval_df)
eval_dataset

Dataset({
    features: ['user_input', 'reference', 'qa_context', 'response', 'retrieved_contexts'],
    num_rows: 25
})

In [92]:
# Huggingface에 upload
eval_dataset.push_to_hub("wiki_olympic_rag_eval_dataset")	# dataset_id : 계정/id

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/mingyu-oo/wiki_olympic_rag_eval_dataset/commit/9925a870fdf1236796a3acff1dc4e6c9b958fca5', commit_message='Upload dataset', commit_description='', oid='9925a870fdf1236796a3acff1dc4e6c9b958fca5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/mingyu-oo/wiki_olympic_rag_eval_dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='mingyu-oo/wiki_olympic_rag_eval_dataset'), pr_revision=None, pr_num=None)

In [93]:
from datasets import load_dataset

# Login using e.g. 'huggingface-cli login' to access this dataset
load_eval_dataset = load_dataset("mingyu-oo/wiki_olympic_rag_eval_dataset")
load_eval_dataset

README.md:   0%|          | 0.00/438 [00:00<?, ?B/s]

c:\Users\Playdata\miniconda3\envs\lang_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\datasets--mingyu-oo--wiki_olympic_rag_eval_dataset. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. 

train-00000-of-00001.parquet:   0%|          | 0.00/34.0k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['user_input', 'reference', 'qa_context', 'response', 'retrieved_contexts'],
        num_rows: 25
    })
})